In [ ]:
NOTEBOOK_VERSION = 'v4-llm-split-beat-sync-render'
print(f'KoenigsbergStories {NOTEBOOK_VERSION}', flush=True)

import subprocess, sys

def run(cmd):
    subprocess.run(cmd, check=True)

run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow', 'opencv-python-headless', 'librosa', 'soundfile', 'requests', 'telethon', 'cryptography'])


In [ ]:
import asyncio, os, shutil, subprocess, sys, threading
from pathlib import Path

def run_async(coro):
    result = None
    error = None
    def runner():
        nonlocal result, error
        try:
            result = asyncio.run(coro)
        except BaseException as exc:
            error = exc
    thread = threading.Thread(target=runner)
    thread.start()
    thread.join()
    if error is not None:
        raise error
    return result

def ensure_story_publish_helper(source_folder):
    helper_path = source_folder / 'kaggle_common' / 'story_publish.py'
    if not helper_path.exists():
        print('[SKIP] common story_publish helper not found in Kenigsberg bundle.', flush=True)
        return False
    target = Path('/kaggle/working/story_publish.py')
    shutil.copy2(helper_path, target)
    if str(target.parent) not in sys.path:
        sys.path.insert(0, str(target.parent))
    return True

input_root = Path('/kaggle/input')
candidates = sorted([p for p in input_root.iterdir() if p.is_dir() and p.name.startswith('kenigsberg-session-')], key=lambda p: p.name, reverse=True)
if not candidates:
    raise RuntimeError('kenigsberg-session-* dataset not found')
bundle = candidates[0]
print(f'Using session bundle: {bundle}', flush=True)
work = Path('/kaggle/working/kenigsberg_runtime')
if work.exists():
    shutil.rmtree(work)
shutil.copytree(bundle, work)
os.chdir(work)
story_publish_requested = (work / 'story_publish.json').exists()
story_publish_ready = ensure_story_publish_helper(work) if story_publish_requested else False
preflight_story_publish_from_kaggle = None
publish_story_from_kaggle = None
if story_publish_ready:
    from story_publish import preflight_story_publish_from_kaggle, publish_story_from_kaggle
if story_publish_requested and preflight_story_publish_from_kaggle is None:
    raise RuntimeError('Kenigsberg story publish requested but shared helper is unavailable')
story_preflight = None
if preflight_story_publish_from_kaggle is not None:
    story_preflight = run_async(preflight_story_publish_from_kaggle(search_roots=[work, input_root], output_dir=Path('/kaggle/working'), log=print))
    if story_preflight and not story_preflight.get('ok'):
        raise RuntimeError(f"Kenigsberg story publish preflight failed: {story_preflight}")
if story_publish_requested and not story_preflight:
    raise RuntimeError('Kenigsberg story publish requested but story_publish.json was not mounted into Kaggle input')
subprocess.run([sys.executable, 'scripts/render_kenigsberg_story.py'], check=True)
final_mp4 = Path('/kaggle/working/kenigsberg_story_final.mp4')
if story_preflight and publish_story_from_kaggle is not None:
    story_report = run_async(publish_story_from_kaggle(final_video_path=final_mp4, intro_path=None, posters=None, search_roots=[work, input_root], output_dir=Path('/kaggle/working'), log=print))
    if story_report and not story_report.get('ok'):
        raise RuntimeError(f"Kenigsberg story publish failed: {story_report}")
